In [ ]:
import pandas as pd
import numpy as np
import joblib

import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.append(str(project_root))



from src.data import clean_feature_names

In [11]:
X_train = pd.read_parquet('../../data/train_data/df_train_final.parquet')
# X_test = pd.read_parquet('../../data/train_data/df_test_final.parquet')

In [3]:
X_num = X_train.select_dtypes(include=["number"])
corr = X_num.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

In [4]:
upper

,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,CC_NAME_CONTRACT_STATUS_SIGNED_COUNT,INSTALLMENTS_AMT_PAYMENT_MEAN,INSTALLMENTS_AMT_PAYMENT_SUM,INSTALLMENTS_AMT_PAYMENT_MAX,INSTALLMENTS_AMT_PAYMENT_MIN,INSTALLMENTS_DAYS_ENTRY_PAYMENT_MEAN,INSTALLMENTS_DAYS_ENTRY_PAYMENT_MIN,INSTALLMENTS_DAYS_ENTRY_PAYMENT_MAX,INSTALLMENTS_NUM_INSTALMENT_NUMBER_MAX,INSTALLMENTS_PAYMENT_TO_INSTALMENT_RATIO_MEAN
SK_ID_CURR,NaN,0.002108,0.001129,0.001820,0.000343,0.000433,0.000232,0.000849,0.001500,0.001366,...,0.000288,0.000043,0.002289,0.000581,0.001889,0.001176,0.001634,0.002708,0.002089,0.004703
TARGET,NaN,NaN,0.019187,0.003982,0.030369,0.012817,0.039645,0.037227,0.078239,0.044932,...,0.006201,0.023169,0.024375,0.001554,0.025724,0.043992,0.058794,0.002298,0.006304,0.001366
CNT_CHILDREN,NaN,NaN,NaN,0.012882,0.002145,0.021374,0.001827,0.025573,0.330938,0.239818,...,0.006452,0.024342,0.052369,0.018870,0.013569,0.006075,0.005694,0.009338,0.020695,0.002170
AMT_INCOME_TOTAL,NaN,NaN,NaN,NaN,0.156870,0.191657,0.159610,0.074796,0.027261,0.064223,...,0.000449,0.079131,0.087306,0.070261,0.024174,0.002086,0.012185,0.007801,0.019954,0.003048
AMT_CREDIT,NaN,NaN,NaN,NaN,NaN,0.770138,0.986968,0.099738,0.055436,0.066838,...,0.017324,0.159906,0.168994,0.172111,0.050447,0.084072,0.088237,0.038383,0.060382,0.002046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
INSTALLMENTS_DAYS_ENTRY_PAYMENT_MEAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.818195,0.664832,0.193133,0.004967
INSTALLMENTS_DAYS_ENTRY_PAYMENT_MIN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.234491,0.396173,0.002737
INSTALLMENTS_DAYS_ENTRY_PAYMENT_MAX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.183622,0.004653
INSTALLMENTS_NUM_INSTALMENT_NUMBER_MAX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004550


In [7]:
upper.to_parquet('../../data/train_data/corr.parquet')


In [12]:
# Load 
upper = pd.read_parquet('../../data/train_data/corr.parquet')
model = joblib.load("../../models/lgbm_model.pkl")
X_train = clean_feature_names(X_train.drop(columns=['TARGET', 'SK_ID_CURR']))
importances = model.feature_importances_
importance_df = pd.Series(importances, index=X_train.columns)

to_drop = []

for col in upper.columns:
    correlated = upper.index[upper[col] > 0.9].tolist()
    
    for corr_col in correlated:
        if importance_df[col] < importance_df[corr_col]:
            to_drop.append(col)
        else:
            to_drop.append(corr_col)

print(f"Nombre de colonnes à supprimer : {len(to_drop)}")
print(to_drop[:20])

Nombre de colonnes à supprimer : 1042
['AMT_GOODS_PRICE', 'FLAG_EMP_PHONE', 'REGION_RATING_CLIENT', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'APARTMENTS_MODE', 'LIVINGAPARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_MODE', 'LIVINGAPARTMENTS_AVG', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_AVG']


In [13]:
len(to_drop), len(set(to_drop))

(1042, 302)

In [14]:
set(to_drop)

{'AMT_GOODS_PRICE',
 'AMT_REQ_CREDIT_BUREAU_DAY_MISSING_FLAG',
 'AMT_REQ_CREDIT_BUREAU_HOUR_MISSING_FLAG',
 'AMT_REQ_CREDIT_BUREAU_MON_MISSING_FLAG',
 'AMT_REQ_CREDIT_BUREAU_QRT_MISSING_FLAG',
 'AMT_REQ_CREDIT_BUREAU_WEEK_MISSING_FLAG',
 'APARTMENTS_AVG_MISSING_FLAG',
 'APARTMENTS_MEDI',
 'APARTMENTS_MEDI_MISSING_FLAG',
 'APARTMENTS_MODE',
 'APARTMENTS_MODE_MISSING_FLAG',
 'BASEMENTAREA_AVG',
 'BASEMENTAREA_AVG_MISSING_FLAG',
 'BASEMENTAREA_MEDI',
 'BASEMENTAREA_MODE_MISSING_FLAG',
 'BB_COUNT_SEVERE_DPD_sum',
 'CC_AMT_BALANCE_MIN',
 'CC_AMT_BALANCE_SUM',
 'CC_AMT_CREDIT_LIMIT_ACTUAL_MAX',
 'CC_AMT_DRAWINGS_ATM_CURRENT_MISSING_FLAG_MEAN',
 'CC_AMT_DRAWINGS_ATM_CURRENT_MISSING_FLAG_MIN',
 'CC_AMT_DRAWINGS_ATM_CURRENT_MISSING_FLAG_SUM',
 'CC_AMT_DRAWINGS_CURRENT_SUM',
 'CC_AMT_DRAWINGS_OTHER_CURRENT_MISSING_FLAG_MAX',
 'CC_AMT_DRAWINGS_OTHER_CURRENT_MISSING_FLAG_MEAN',
 'CC_AMT_DRAWINGS_OTHER_CURRENT_MISSING_FLAG_MIN',
 'CC_AMT_DRAWINGS_OTHER_CURRENT_MISSING_FLAG_SUM',
 'CC_AMT_DRAWINGS_P